In [29]:
import pandas as pd

# Load the dataset to examine its contents and structure
file_path = 'Advertising.csv'
df = pd.read_csv(file_path)

# Display the first few rows to understand the dataset
df.head()


,Unnamed: 0,TV,radio,newspaper,sales
0,1,230.1,37.8,69.2,22.1
1,2,44.5,39.3,45.1,10.4
2,3,17.2,45.9,69.3,9.3
3,4,151.5,41.3,58.5,18.5
4,5,180.8,10.8,58.4,12.9


In [30]:
# The "Unnamed: 0" column appears to be the index for each row
# We will convert this into a datetime column where each value represents the start of a new week
# First, rename the column to 'Week'
df.rename(columns={'Unnamed: 0': 'Week'}, inplace=True)

# Convert the 'Week' column into a series of weekly dates starting from a specific date (e.g., "2023-01-01")
# Assuming each row represents a consecutive week
start_date = pd.to_datetime("2020-01-01")  # Starting from the first week of 2023
df['Week'] = pd.date_range(start=start_date, periods=len(df), freq='W-SUN')  # Weekly frequency, ending on Sunday
df['sales'] = df.sales * 1000
# Display the updated dataframe to verify
df.head()


,Week,TV,radio,newspaper,sales
0,2020-01-05,230.1,37.8,69.2,22100.0
1,2020-01-12,44.5,39.3,45.1,10400.0
2,2020-01-19,17.2,45.9,69.3,9300.0
3,2020-01-26,151.5,41.3,58.5,18500.0
4,2020-02-02,180.8,10.8,58.4,12900.0


In [31]:
max(df.Week)

Timestamp('2023-10-29 00:00:00')

In [32]:
# df.to_csv('spend_data.csv')

In [33]:
data = df[['TV', 'radio', 'newspaper','sales']] 

In [34]:
# Step 1: Data Preparation

# Check for missing values and basic statistics
data.info()
data.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   TV         200 non-null    float64
 1   radio      200 non-null    float64
 2   newspaper  200 non-null    float64
 3   sales      200 non-null    float64
dtypes: float64(4)
memory usage: 6.4 KB


,TV,radio,newspaper,sales
count,200.000000,200.000000,200.000000,200.000000
mean,147.042500,23.264000,30.554000,14022.500000
std,85.854236,14.846809,21.778621,5217.456566
min,0.700000,0.000000,0.300000,1600.000000
25%,74.375000,9.975000,12.750000,10375.000000
50%,149.750000,22.900000,25.750000,12900.000000
75%,218.825000,36.525000,45.100000,17400.000000
max,296.400000,49.600000,114.000000,27000.000000


In [35]:
# No missing values detected, proceeding to normalization
# Normalize the feature columns (TV, radio, newspaper) for better model interpretation
from sklearn.preprocessing import MinMaxScaler

# Initialize the scaler
scaler = MinMaxScaler()

# Apply scaling to the ad spend columns
data[['TV', 'radio', 'newspaper']] = scaler.fit_transform(data[['TV', 'radio', 'newspaper']])

# Display the normalized data to verify
data.head()


/var/folders/d7/5lnxp9g912g0c9nbw2ynhg5m0000gn/T/ipykernel_40520/872977631.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[['TV', 'radio', 'newspaper']] = scaler.fit_transform(data[['TV', 'radio', 'newspaper']])


,TV,radio,newspaper,sales
0,0.775786,0.762097,0.605981,22100.0
1,0.148123,0.792339,0.394019,10400.0
2,0.055800,0.925403,0.606860,9300.0
3,0.509976,0.832661,0.511873,18500.0
4,0.609063,0.217742,0.510994,12900.0


In [36]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# Step 2: Define features and target
X = data[['TV', 'radio', 'newspaper']]  # Independent variables (ad spend channels)
y = data['sales']  # Dependent variable (sales)

# Step 3: Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 4: Build the linear regression model
model = LinearRegression()
model.fit(X_train, y_train)

# Step 5: Make predictions on the test set
y_pred = model.predict(X_test)

# Calculate performance metrics (e.g., RMSE)
rmse = mean_squared_error(y_test, y_pred, squared=False)

# Step 6: Extract the coefficients to interpret the ROI for each channel
coefficients = model.coef_
intercept = model.intercept_

# Display results
rmse, coefficients, intercept


(1781.5996615334504,
 array([13226.5183155 ,  9384.07469003,   313.93870061]),
 3011.2063346531468)

In [37]:
# Step 3: Calculate the sales generated by each channel
sales_generated = total_spend * coefficients

# Step 4: Calculate ROI for each channel
roi = (sales_generated - total_spend) / total_spend

# Step 5: Create a DataFrame to present the results
roi_df = pd.DataFrame({
    'Channel': ['TV', 'Radio', 'Newspaper'],
    'Total Spend': total_spend,
    'Sales Generated': sales_generated,
    'ROI': roi
})

In [38]:
roi

array([13225.5183155 ,  9383.07469003,   312.93870061])

In [39]:
roi_df

,Channel,Total Spend,Sales Generated,ROI
0,TV,1,13226.518315,13225.518315
1,Radio,1,9384.074690,9383.074690
2,Newspaper,1,313.938701,312.938701


In [14]:
# Step 7: ROI Calculation

# The coefficients give us the marginal increase in sales per unit increase in ad spend.
# Let's calculate the ROI for each channel based on their coefficients and normalized spend.

# Calculate the total ad spend (since the data is normalized, we will assume total spend to be 1 unit)
total_spend = 1

# Proportion of total spend by channel (current budget distribution)
spend_distribution = X.mean()

# Calculate current sales impact (contribution from each channel)
sales_contribution = coefficients * spend_distribution

# ROI for each channel: sales_contribution divided by the proportional spend
roi = sales_contribution / spend_distribution

# Step 8: Insights on optimized spend
# We'll try different allocation strategies to maximize sales.

# Hypothetical budget distribution scenarios (e.g., increasing/decreasing spend on TV)
scenarios = {
    'Current Distribution': spend_distribution,
    'Increase TV Spend': [0.8, 0.6, 0.2],
    'Increase Radio Spend': [0.6, 0.9, 0.2],
    'Newspaper Spend': [0.2, 0.5, 0.9],
}

# Calculate predicted sales for each scenario
optimized_sales = {}
for scenario, allocation in scenarios.items():
    predicted_sales = intercept + sum(c * a for c, a in zip(coefficients, allocation))
    optimized_sales[scenario] = predicted_sales

# Display ROI and optimized spend predictions
roi, optimized_sales


(TV           13.226518
 radio         9.384075
 newspaper     0.313939
 dtype: float64,
 {'Current Distribution': 14.04200423981102,
  'Increase TV Spend': 19.285653541190413,
  'Increase Radio Spend': 19.45557228509805,
  'Newspaper Spend': 10.631092173317668})

In [13]:
spend_distribution

TV           0.494902
radio        0.469032
newspaper    0.266086
dtype: float64